# PathSnap: first review

This notebook runs locally on the Ubuntu laptop. It calls the readable source scripts in `src/` so the demo uses the same code as the command line. Run cells in order. The processed candidate tables are reusable by every teammate.

In [ ]:
from pathlib import Path
import subprocess, json, csv, numpy as np, sys

# Work whether Jupyter starts in this folder or in the repository root.
PROJECT = Path.cwd().resolve()
if not (PROJECT / 'src' / 'prepare_shared_data.py').exists():
    bundle = PROJECT / 'PES2UG24CS102_RandomForest'
    if (bundle / 'src' / 'prepare_shared_data.py').exists():
        PROJECT = bundle
assert (PROJECT / 'src' / 'prepare_shared_data.py').exists(), 'Open this notebook from the PES2UG24CS102_RandomForest folder or its repository root.'
PYTHON = sys.executable
def run_script(name):
    subprocess.run([PYTHON, str(PROJECT / 'src' / name)], cwd=PROJECT, check=True)
print('Project folder:', PROJECT)
print('Python interpreter:', PYTHON)


## 1. Audit the provided raw files

The audit counts trips, observations, network rows, timestamp gaps, and route connectivity. It does not change the data.

In [ ]:
run_script('eda.py')
with (PROJECT / 'results/trip_inventory.csv').open() as f:
    for row in csv.DictReader(f):
        if row['trip'] == '00000001':
            print('Example trip inventory:', row)

## 2. Create the shared candidate-feature tables

For each GPS observation, preprocessing finds nearby directed road arcs and computes distance, heading, along-segment position, segment length, GPS speed, and time gap. The `.route` file supplies a pointwise geometric label. `candidate_miss_rate` is recorded separately.

In [ ]:
run_script('prepare_shared_data.py')
for part in ('train', 'student_test'):
    print(part, json.loads((PROJECT / f'shared_data/{part}_metadata.json').read_text()))
z = np.load(PROJECT / 'shared_data/train_candidates.npz', allow_pickle=False)
print('First candidate feature row:', dict(zip(z['feature_names'].astype(str), z['X'][0])))
print('Target, trip, observation, candidate arc:', z['y'][0], z['trip_id'][0], z['gps_observation_id'][0], z['candidate_arc_id'][0])

## 3. Train and evaluate Random Forest

Five-fold GroupKFold keeps all candidate rows from each trip in the same fold. The student-test partition is used only for development evaluation. The saved model is written to `artifacts/random_forest.joblib`.

In [ ]:
run_script('train_random_forest.py')
metrics = json.loads((PROJECT / 'results/random_forest_results.json').read_text())
for key in ('grouped_5fold_cv_observation_top1_mean', 'student_test_observation_top1', 'train_candidate_miss_rate', 'student_test_candidate_miss_rate', 'final_fit_seconds'):
    print(f'{key}: {metrics[key]}')

## 4. Inspect a real trip on a map

Blue shows the observed GPS track; green shows the supplied reference route. The route is for development inspection only.

In [ ]:
run_script('make_map.py')
print('Open this HTML file:', PROJECT / 'figures/trip_00000001_reference.html')